# Share torque between a bonded core and a sleeve

The core and the sleeve have equal twist per length because they are bonded. Predict which material carries more torque before editing the radii or either shear modulus. Run all cells after changing an input.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

a, b = .012, .020      # core and outer radii, m
Gc, Gs = 79e9, 27e9   # core and sleeve shear moduli, Pa
torque, length = 100., .60   # N m, m
assert 0 < a < b and min(Gc,Gs,length) > 0

## Compatibility sets the twist; equilibrium sets its magnitude
The stresses differ across the material interface even though the tangential displacement is continuous. Check the torque independently by integrating the lever-arm weighted shear traction over the end face.

In [ ]:
Jc,Js=np.pi*a**4/2,np.pi*(b**4-a**4)/2
twist=torque/(Gc*Jc+Gs*Js)
Tc,Ts=Gc*Jc*twist,Gs*Js*twist
r1,r2=np.linspace(0,a,140),np.linspace(a,b,140)
def integrate(y,x):
    return np.sum(np.diff(x)*(y[1:]+y[:-1])/2)
Tnum=integrate(2*np.pi*r1**2*Gc*twist*r1,r1)+integrate(2*np.pi*r2**2*Gs*twist*r2,r2)
assert np.isclose(Tnum,torque,rtol=1e-4)
print(f"Torque shares {Tc:.2f} and {Ts:.2f} N m; end rotation {np.rad2deg(twist*length):.3f} degrees")
print(f"Shear just inside/outside the interface: {Gc*a*twist/1e6:.2f} / {Gs*a*twist/1e6:.2f} MPa")

## Distinguish a local stress peak from a total torque
The left panel is the actual section. The right panel sweeps the core radius at constant outer radius and material properties.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(9,3.5),layout="constrained")
ax[0].plot(r1*1e3,Gc*twist*r1/1e6,label="Core")
ax[0].plot(r2*1e3,Gs*twist*r2/1e6,label="Sleeve")
ax[0].set(xlabel="Radius (mm)",ylabel="End-face shear stress (MPa)",title="Bonded cross section")
fraction=np.linspace(.01,.99,250)
share=Gc*fraction**4/(Gc*fraction**4+Gs*(1-fraction**4))
ax[1].plot(fraction,share,label="Core share")
ax[1].plot(fraction,1-share,"--",label="Sleeve share")
ax[1].set(xlabel="Core / outer radius",ylabel="Fraction of torque",title="Changing the core size")
for p in ax: p.legend(fontsize=8);p.grid(alpha=.22)
plt.show()

Set the two moduli equal. Does the stress jump persist? Find the radius fraction for equal torque share and verify by changing the input.